<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-11-self-hosting/lesson-11.2-custom-fastapi/practice/GCP_Capstone_11.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 11.2 — Custom FastAPI + vLLM

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: install validation deps + Google Cloud auth

This notebook **authors and structure-checks** the source files for a production vLLM inference server. The heavy runtime deps (`vllm`, `torch`, `presidio`) live in the Docker image built in Exercise 8 — in Colab we only install the light libraries needed to validate the FastAPI/Pydantic/OpenAI-SDK code. Run these two cells first.

In [ ]:
%%bash
# Light deps for validating the server code in Colab (vllm/presidio ship in the container).
pip install -q "fastapi>=0.135.0" "pydantic>=2.8.0" "openai>=1.0.0" \
    "google-cloud-bigquery>=3.25.0" "google-cloud-firestore>=2.16.0" \
    "slowapi>=0.1.9" "httpx>=0.27.0"
echo "deps installed"

In [ ]:
# Google Cloud auth — Application Default Credentials, never API keys.
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab auth OK (ADC active)")
except ImportError:
    print("Not on Colab — assuming ADC via `gcloud auth application-default login`")

PROJECT_ID = "documind-ai-YOUR-ID"   # replace with your project id
REGION     = "us-central1"           # asia-south1 for India production
MODEL_NAME = "google/gemma-3-4b-it"  # served by vLLM inside the container
print(f"Project={PROJECT_ID}  Region={REGION}  Model={MODEL_NAME}")

## Exercise 1: FastAPI app skeleton

**Difficulty:** Easy

Lifespan context that loads `AsyncLLMEngine` + exposes a `/health` liveness endpoint. Start uvicorn locally (without GPU, just to validate structure).

**Steps:**
1. Write an `@asynccontextmanager` lifespan that builds `AsyncEngineArgs` and loads `AsyncLLMEngine.from_engine_args(...)` on startup, releasing it on shutdown.
2. Create the `FastAPI(...)` app wired to that lifespan.
3. Add a `/health` route that returns `{"status": "alive"}`.
4. Validate the module parses cleanly (no GPU needed to check structure).

In [ ]:
MAIN_PY = '''
# main.py - Production FastAPI + vLLM server
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request, HTTPException, Depends, BackgroundTasks, Security
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.security import APIKeyHeader
from fastapi.exceptions import RequestValidationError
from slowapi import Limiter
from slowapi.errors import RateLimitExceeded
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from vllm.sampling_params import SamplingParams
from vllm.utils import random_uuid
import asyncio, json, time, datetime, os, logging

logger = logging.getLogger("documind")

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("Loading model into GPU...")
    engine_args = AsyncEngineArgs(
        model=os.getenv("MODEL_NAME", "google/gemma-3-4b-it"),
        tensor_parallel_size=1,
        gpu_memory_utilization=0.90,
        max_model_len=8192,
        dtype="auto",
        trust_remote_code=True,
    )
    app.state.engine = AsyncLLMEngine.from_engine_args(engine_args)
    app.state.engine_ready = True
    logger.info("Engine loaded. GPU ready.")
    yield
    app.state.engine_ready = False
    app.state.engine = None

app = FastAPI(title="DocuMind AI Inference", version="1.0.0", lifespan=lifespan)

@app.get("/health")
async def health():
    """Liveness: process is up. Does NOT check the GPU engine (see /ready)."""
    return {"status": "alive"}
'''
with open('main.py', 'w') as f:
    f.write(MAIN_PY)

# Structure check without importing vllm: parse the source.
import ast
ast.parse(MAIN_PY)
print('main.py (part 1 of 4) written and parses cleanly')
print('Lifespan loads AsyncLLMEngine on startup, releases on shutdown')
print('/health returns {"status":"alive"}')

## Exercise 2: Pydantic v2 schema

**Difficulty:** Easy

Define `ChatMessage`, `ChatCompletionRequest`, `UsageInfo` matching the OpenAI spec. Validate with the `openai` SDK.

**Steps:**
1. Model request/response with Pydantic v2, using `Literal` role types.
2. Set `model_config = {"extra": "allow"}` so vLLM-specific sampling params pass through.
3. Add the streaming delta + chunk models.
4. Round-trip a request built by the `openai` SDK payload shape to confirm compatibility.

In [ ]:
SCHEMA_PY = '''
# schemas.py - Pydantic v2 models matching OpenAI API
from typing import Literal, Optional, Union
from pydantic import BaseModel, Field
import time, uuid

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant", "tool"]
    content: Optional[str] = None
    name: Optional[str] = None

class ChatCompletionRequest(BaseModel):
    model: str
    messages: list[ChatMessage]
    temperature: Optional[float] = Field(default=1.0, ge=0.0, le=2.0)
    top_p: Optional[float] = Field(default=1.0, ge=0.0, le=1.0)
    max_tokens: Optional[int] = None
    stream: Optional[bool] = False
    stop: Optional[Union[str, list[str]]] = None
    user: Optional[str] = None
    model_config = {"extra": "allow"}

class AssistantMessage(BaseModel):
    role: Literal["assistant"] = "assistant"
    content: Optional[str] = None

class UsageInfo(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

class ChatCompletionChoice(BaseModel):
    index: int
    message: AssistantMessage
    finish_reason: Optional[Literal["stop","length","tool_calls","content_filter"]] = None

class ChatCompletionResponse(BaseModel):
    id: str = Field(default_factory=lambda: f"chatcmpl-{uuid.uuid4().hex[:29]}")
    object: Literal["chat.completion"] = "chat.completion"
    created: int = Field(default_factory=lambda: int(time.time()))
    model: str
    choices: list[ChatCompletionChoice]
    usage: UsageInfo

# Streaming delta model
class DeltaMessage(BaseModel):
    role: Optional[Literal["assistant"]] = None
    content: Optional[str] = None

class ChatCompletionStreamChoice(BaseModel):
    index: int
    delta: DeltaMessage
    finish_reason: Optional[str] = None

class ChatCompletionChunk(BaseModel):
    id: str
    object: Literal["chat.completion.chunk"] = "chat.completion.chunk"
    created: int
    model: str
    choices: list[ChatCompletionStreamChoice]
'''
with open('schemas.py', 'w') as f:
    f.write(SCHEMA_PY)
print('schemas.py written')
print('Pydantic v2 models matching OpenAI Chat Completions API exactly')

In [ ]:
# Validate: the schema accepts an openai-SDK-shaped payload (only pydantic needed here).
from schemas import ChatCompletionRequest

req = ChatCompletionRequest.model_validate({
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "You are a document assistant."},
        {"role": "user", "content": "Summarize this invoice."},
    ],
    "temperature": 0.2,
    "stream": False,
    # extra vLLM param flows through thanks to extra='allow'
    "structured_outputs": {"json": {"type": "object"}},
})
print("parsed OK ->", req.model)
print("roles:", [m.role for m in req.messages])
print("passthrough extra kept:", req.model_dump().get("structured_outputs"))

## Exercise 3: Readiness endpoint

**Difficulty:** Easy

`/ready` that checks the `engine_ready` flag + calls `engine.check_health()`. Returns 503 if not ready.

**Steps:**
1. Read `app.state.engine_ready`; return 503 while the model is still loading (cold start).
2. Once ready, call `engine.check_health()` to confirm the GPU worker is responsive.
3. Return 200 `{"status":"ready"}` only when both pass.
4. Keep it separate from `/health` — Cloud Run uses liveness vs. readiness differently.

In [ ]:
# Fresh glue: the practice lab points at "Step 6 code" (no dedicated notebook cell).
# This readiness route is appended to main.py; the app + Request import already exist there.
READY_PY = '''
@app.get("/ready")
async def ready(request: Request):
    """Readiness: 503 during cold start / unhealthy engine, 200 once GPU is serving."""
    if not getattr(request.app.state, "engine_ready", False):
        raise HTTPException(status_code=503, detail="Engine still loading")
    try:
        await request.app.state.engine.check_health()
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Engine unhealthy: {e}")
    return {"status": "ready"}
'''
with open('main.py', 'a') as f:
    f.write(READY_PY)

import ast
ast.parse(READY_PY)
print('/ready appended to main.py and parses cleanly')
print('Cold start -> 503; after engine loads + check_health passes -> 200')

## Exercise 4: SSE streaming generator

**Difficulty:** Medium

Async generator converting vLLM `RequestOutput` to the OpenAI chunk format, terminated with `[DONE]`. Handle client disconnects.

**Steps:**
1. Iterate `engine.generate(...)` and emit one `chat.completion.chunk` per delta.
2. Send an opening role chunk first, then content chunks, then a final `finish_reason` chunk.
3. On each iteration check `request.is_disconnected()` — if the client left, call `engine.abort(request_id)` to free the GPU.
4. Emit `data: [DONE]` at the end; handle `asyncio.CancelledError` by aborting.

In [ ]:
STREAMING_PY = '''
import asyncio, json, time
from fastapi import Request

async def generate_sse_stream(engine, request_id, model_name, prompt, sampling_params, request: Request):
    """Convert vLLM RequestOutput stream to OpenAI SSE format."""
    created = int(time.time())
    first_chunk = True
    prev_len = {}  # completion index -> chars already sent
    try:
        async for output in engine.generate(prompt, sampling_params, request_id):
            # Client disconnected? Abort and free GPU
            if await request.is_disconnected():
                await engine.abort(request_id)
                return
            
            for completion in output.outputs:
                if first_chunk:
                    chunk = {"id": request_id, "object": "chat.completion.chunk",
                             "created": created, "model": model_name,
                             "choices": [{"index": 0,
                                "delta": {"role": "assistant", "content": ""},
                                "finish_reason": None}]}
                    yield f"data: {json.dumps(chunk)}\\n\\n"
                    first_chunk = False
                
                # vLLM completion.text is CUMULATIVE -> emit only the new suffix
                idx = completion.index
                sent = prev_len.get(idx, 0)
                new_text = completion.text[sent:]
                prev_len[idx] = len(completion.text)
                if new_text:
                    chunk = {"id": request_id, "object": "chat.completion.chunk",
                             "created": created, "model": model_name,
                             "choices": [{"index": 0,
                                "delta": {"content": new_text},
                                "finish_reason": None}]}
                    yield f"data: {json.dumps(chunk)}\\n\\n"
                
                if completion.finish_reason is not None:
                    chunk = {"id": request_id, "object": "chat.completion.chunk",
                             "created": created, "model": model_name,
                             "choices": [{"index": 0, "delta": {},
                                "finish_reason": completion.finish_reason}]}
                    yield f"data: {json.dumps(chunk)}\\n\\n"
        
        yield "data: [DONE]\\n\\n"
    except asyncio.CancelledError:
        await engine.abort(request_id)
        raise
'''
with open('streaming.py', 'w') as f:
    f.write(STREAMING_PY)
print('streaming.py written')
print('CRITICAL: SSE headers needed when mounting:')
print('  Content-Type: text/event-stream')
print('  Cache-Control: no-cache')
print('  X-Accel-Buffering: no  (disable proxy buffering)')

## Exercise 5: Per-tenant auth

**Difficulty:** Medium

`APIKeyHeader` dependency + `get_tenant()` lookup in Firestore (hashed keys) + SlowAPI rate limiter.

**Steps:**
1. Read the key from the `X-API-Key` header; hash it (SHA-256) before any lookup — never store plaintext.
2. Look the hash up in Firestore; 401 on miss or expired key (expiry enables key rotation).
3. Fetch the tenant doc to get its `tier`.
4. Wire a SlowAPI `Limiter` keyed on the API key, with tiered limits (free/pro/enterprise).

In [ ]:
AUTH_PY = '''
import hashlib
import datetime
import os
from fastapi import Depends, HTTPException, Security, Request
from fastapi.security import APIKeyHeader
from slowapi import Limiter
from google.cloud import firestore

api_key_header = APIKeyHeader(name="X-API-Key")
firestore_client = firestore.Client(project=os.environ.get("GOOGLE_CLOUD_PROJECT"))

def hash_api_key(plaintext: str) -> str:
    """SHA-256 for lookup; for production use bcrypt with salt."""
    return hashlib.sha256(plaintext.encode()).hexdigest()

async def get_tenant(api_key: str = Security(api_key_header)) -> dict:
    """Lookup tenant by HASHED key. Supports key rotation via multiple active keys."""
    key_hash = hash_api_key(api_key)
    doc = firestore_client.collection("api_keys").document(key_hash).get()
    if not doc.exists:
        raise HTTPException(status_code=401, detail="Invalid API key")
    
    data = doc.to_dict()
    # Check expiry for key rotation
    if data.get("expires_at") and data["expires_at"] < datetime.datetime.utcnow():
        raise HTTPException(status_code=401, detail="Expired API key")
    
    # Fetch tenant details
    tenant_doc = firestore_client.collection("tenants").document(data["tenant_id"]).get()
    return tenant_doc.to_dict()  # {tenant_id, tier, rate_limit}

def rate_limit_key(request: Request):
    return request.headers.get("X-API-Key", "anonymous")

limiter = Limiter(key_func=rate_limit_key)

def tier_rate_limit(request: Request):
    tier = getattr(request.state, "tenant_tier", "free")
    return {"free": "10/minute",
            "pro": "100/minute",
            "enterprise": "1000/minute"}.get(tier, "10/minute")
'''
with open('auth.py', 'w') as f:
    f.write(AUTH_PY)
print('auth.py written')
print('API keys HASHED (never plaintext) in Firestore')
print('Multiple active keys per tenant with expiry for rotation')
print('Tiered rate limits: free 10/min, pro 100/min, enterprise 1000/min')

## Exercise 6: BigQuery + PII redaction

**Difficulty:** Medium

`BackgroundTasks` async log with Presidio PII scrub. Create a BigQuery table partitioned by date and clustered by `tenant_id`.

**Steps:**
1. Redact prompts with Presidio NER, plus a regex safety net for email/phone/SSN.
2. Stream one row per request into BigQuery via a non-blocking insert (fire it from `BackgroundTasks`).
3. Create the `inference_logs` table `PARTITION BY DATE(timestamp)` and `CLUSTER BY tenant_id, model`.
4. Confirm you can query per-tenant cost/volume in SQL.

In [ ]:
LOGGING_PY = '''
import re, datetime
from google.cloud import bigquery
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

pii_analyzer = AnalyzerEngine()
pii_anonymizer = AnonymizerEngine()
bq_client = bigquery.Client()
BQ_TABLE = "project.dataset.inference_logs"

def redact_pii(text: str) -> str:
    """Presidio NER + regex fallback."""
    try:
        results = pii_analyzer.analyze(text=text, language="en")
        if results:
            return pii_anonymizer.anonymize(text=text, analyzer_results=results).text
    except Exception:
        pass
    # Regex safety net
    text = re.sub(r"\\b[\\w.-]+@[\\w.-]+\\.\\w+\\b", "<EMAIL>", text)
    text = re.sub(r"\\b\\d{3}[-.]?\\d{3}[-.]?\\d{4}\\b", "<PHONE>", text)
    text = re.sub(r"\\b\\d{3}-\\d{2}-\\d{4}\\b", "<SSN>", text)
    return text

async def log_request(entry: dict):
    """Non-blocking BigQuery streaming insert."""
    errors = bq_client.insert_rows_json(BQ_TABLE, [entry])
    if errors:
        print(f"BQ insert failed: {errors}")
'''
with open('logging_module.py', 'w') as f:
    f.write(LOGGING_PY)
print('logging_module.py written')
print('Presidio NER + regex fallback scrubs prompts before they hit BigQuery')

In [ ]:
%%bash
# Create the partitioned + clustered log table (DDL grounded in the notebook's Cell 5 schema).
# Replace DATASET; the table stays cheap to query per-tenant thanks to clustering.
bq query --use_legacy_sql=false '
CREATE TABLE IF NOT EXISTS `DATASET.inference_logs` (
  request_id       STRING,
  tenant_id        STRING,
  model            STRING,
  prompt_tokens    INT64,
  completion_tokens INT64,
  latency_ms       FLOAT64,
  status_code      INT64,
  timestamp        TIMESTAMP,
  prompt_preview   STRING
)
PARTITION BY DATE(timestamp)
CLUSTER BY tenant_id, model;
'

## Exercise 7: Guided JSON classify endpoint

**Difficulty:** Challenge

`/v1/documind/classify` with `StructuredOutputsParams` + Pydantic schema enforcement. Test with 50 documents.

**Steps:**
1. Define a `ClassificationResult` Pydantic model (category `Literal`, confidence, reasoning).
2. Pass its JSON schema to `StructuredOutputsParams(json=...)` on the `SamplingParams`.
3. Run generation at `temperature=0.0`; structured outputs enforce valid JSON at the token level.
4. `model_validate_json` the output — no retry-on-parse-error logic is ever needed.

In [ ]:
DOCUMIND_PY = '''
from pydantic import BaseModel
from typing import Literal
from fastapi import APIRouter, Request, Depends
from vllm.sampling_params import SamplingParams, StructuredOutputsParams
from vllm.utils import random_uuid
import json
from auth import get_tenant  # get_tenant is defined in auth.py

router = APIRouter(prefix="/v1/documind", tags=["documind"])

class ClassificationResult(BaseModel):
    category: Literal["invoice","contract","report","letter","other"]
    confidence: float
    reasoning: str

# Scalar params bind as QUERY string; wrap inputs in a model so FastAPI
# reads the POSTed JSON body (a bare `text: str` would 422 on a JSON body).
class ClassifyRequest(BaseModel):
    text: str

class ExtractRequest(BaseModel):
    text: str
    schema_def: dict  # client-provided JSON Schema

@router.post("/classify")
async def classify_document(req: ClassifyRequest, request: Request, tenant: dict = Depends(get_tenant)):
    so = StructuredOutputsParams(json=ClassificationResult.model_json_schema())
    sampling = SamplingParams(temperature=0.0, max_tokens=256, structured_outputs=so)
    prompt = f"Classify this document into: invoice, contract, report, letter, other.\\n\\nDocument:\\n{req.text[:4000]}\\n\\nRespond with JSON:"
    
    request_id = random_uuid()
    final = None
    async for output in request.app.state.engine.generate(prompt, sampling, request_id):
        final = output
    # Structured outputs GUARANTEE valid JSON matching schema
    return ClassificationResult.model_validate_json(final.outputs[0].text)

@router.post("/extract")
async def extract_fields(req: ExtractRequest, request: Request, tenant: dict = Depends(get_tenant)):
    so = StructuredOutputsParams(json=req.schema_def)
    sampling = SamplingParams(temperature=0.0, max_tokens=1024, structured_outputs=so)
    prompt = f"Extract structured data matching the schema.\\n\\nDocument:\\n{req.text[:4000]}\\n\\nJSON:"
    
    request_id = random_uuid()
    final = None
    async for output in request.app.state.engine.generate(prompt, sampling, request_id):
        final = output
    return json.loads(final.outputs[0].text)
'''
with open('documind.py', 'w') as f:
    f.write(DOCUMIND_PY)
print('documind.py written')
print('/v1/documind/classify - 100% valid JSON matching ClassificationResult')
print('/v1/documind/extract - 100% valid JSON matching client schema')
print('NO retry-on-parse-error logic needed - structured outputs enforce at token level')

## Exercise 8: Deploy + load test

**Difficulty:** Challenge

Dockerfile, Cloud Run deploy, then measure streaming TTFT and p95 latency with 10 concurrent clients.

**Steps:**
1. Build on the `vllm/vllm-openai` base image; install app deps; run uvicorn with `--workers 1` (GPU memory is not shareable across processes).
2. Deploy to Cloud Run with a GPU (`--gpu 1 --gpu-type nvidia-l4`), `--concurrency 10`, `--min-instances 1`.
3. Point the standard `openai` SDK at the service `base_url`; use `httpx` + `tenacity` for custom endpoints with cold-start retry.
4. Fire 10 concurrent streaming clients and report mean/p95 TTFT and total latency.

In [ ]:
DOCKERFILE = '''
FROM vllm/vllm-openai:v0.28.0

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

ENV PORT=8080
EXPOSE 8080

# Reset the vllm-openai base image ENTRYPOINT so our uvicorn CMD runs as the command
ENTRYPOINT []

# --workers 1 MANDATORY - GPU memory not shareable across processes
CMD ["python", "-m", "uvicorn", "main:app", \\
     "--host", "0.0.0.0", "--port", "8080", \\
     "--workers", "1", "--timeout-keep-alive", "120"]
'''
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)

REQUIREMENTS = '''
fastapi>=0.135.0
uvicorn[standard]>=0.34.0
slowapi>=0.1.9
pydantic>=2.8.0
google-cloud-bigquery>=3.25.0
google-cloud-firestore>=2.16.0
google-cloud-logging>=3.11.0
google-cloud-secret-manager>=2.20.0
opentelemetry-sdk>=1.27.0
opentelemetry-exporter-gcp-trace>=1.7.0
opentelemetry-instrumentation-fastapi>=0.48b0
presidio-analyzer>=2.2.0
presidio-anonymizer>=2.2.0
httpx>=0.27.0
'''
with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)

print('Dockerfile + requirements.txt written')
print('Base: vllm/vllm-openai:v0.28.0 (CUDA + PyTorch + vLLM)')
print('+ FastAPI + google-cloud + Presidio + OpenTelemetry')
print('Final image size: ~14GB')

In [ ]:
%%bash
# Deploy to Cloud Run with an L4 GPU. Source deploy builds the Dockerfile above.
# min-instances 1 keeps one warm engine so TTFT stays low (no cold-start on every call).
gcloud run deploy documind-inference --source . \
  --region us-central1 \
  --gpu 1 --gpu-type nvidia-l4 --cpu 8 --memory 32Gi \
  --concurrency 10 --max-instances 3 --min-instances 1 \
  --no-cpu-throttling --no-gpu-zonal-redundancy

In [ ]:
CLIENT_PY = '''
from openai import OpenAI
import httpx
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

SERVICE_URL = "https://documind-inference-xxxxx.run.app"
API_KEY = "sk-tenant-abc123"

# Standard OpenAI SDK works unchanged
client = OpenAI(
    base_url=f"{SERVICE_URL}/v1",
    api_key=API_KEY,
    default_headers={"X-API-Key": API_KEY},
)

# Non-streaming
response = client.chat.completions.create(
    model="google/gemma-3-4b-it",
    messages=[{"role": "user", "content": "Summarize this contract..."}],
    max_tokens=500,
)
print(response.choices[0].message.content)

# Streaming
stream = client.chat.completions.create(
    model="google/gemma-3-4b-it",
    messages=[{"role": "user", "content": "Explain this invoice..."}],
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

# Custom DocuMind endpoint with cold start retry
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    retry=retry_if_exception_type((httpx.ConnectError, httpx.ReadTimeout)),
)
async def classify_document(text: str) -> dict:
    async with httpx.AsyncClient(timeout=120.0) as client:
        response = await client.post(
            f"{SERVICE_URL}/v1/documind/classify",
            headers={"X-API-Key": API_KEY},
            json={"text": text},
        )
        if response.status_code == 503:
            raise httpx.ReadTimeout("Cold start - retrying")
        response.raise_for_status()
        return response.json()
'''
with open('client.py', 'w') as f:
    f.write(CLIENT_PY)
print('client.py written')
print('Standard openai SDK works unchanged - just change base_url')
print('For custom endpoints: httpx + tenacity for retry-on-cold-start')

In [ ]:
# Fresh load test (the notebook stops at client patterns). Measures streaming TTFT + total
# latency across N concurrent clients. Set SERVICE_URL/API_KEY from your deploy, then run it.
import asyncio, time, statistics, httpx

SERVICE_URL = "https://documind-inference-xxxxx.run.app"  # from the gcloud deploy output
API_KEY = "sk-tenant-abc123"

def _p95(xs):
    xs = sorted(xs)
    return xs[max(0, int(0.95 * len(xs)) - 1)]

async def one_stream():
    t0 = time.perf_counter()
    ttft = None
    async with httpx.AsyncClient(timeout=120.0) as c:
        async with c.stream(
            "POST", f"{SERVICE_URL}/v1/chat/completions",
            headers={"X-API-Key": API_KEY},
            json={"model": MODEL_NAME,
                  "messages": [{"role": "user", "content": "Summarize this invoice in 3 lines."}],
                  "stream": True, "max_tokens": 128},
        ) as r:
            async for line in r.aiter_lines():
                if ttft is None and line.startswith("data:") and '"content"' in line:
                    ttft = time.perf_counter() - t0
    return ttft or 0.0, time.perf_counter() - t0

async def load_test(n=10):
    results = await asyncio.gather(*[one_stream() for _ in range(n)])
    ttfts = [r[0] for r in results if r[0] > 0]
    totals = [r[1] for r in results]
    print(f"clients={n}")
    print(f"TTFT   mean={statistics.mean(ttfts):.2f}s  p95={_p95(ttfts):.2f}s   (target < 1s warm)")
    print(f"total  mean={statistics.mean(totals):.2f}s  p95={_p95(totals):.2f}s  (target p95 < 3s)")

# In Colab: await load_test(10)   # requires the real SERVICE_URL + API_KEY above
print("Load test ready. After deploy, set SERVICE_URL/API_KEY, then: await load_test(10)")